# 🎓 RAG Training with Calliope AI

This notebook demonstrates how to train Calliope AI using RAG (Retrieval-Augmented Generation) to improve query accuracy and understand your specific business logic.

## What is RAG Training?

RAG training allows you to teach Calliope about:
- **Schema details**: Table structures, relationships, constraints
- **Business logic**: Custom calculations, metrics definitions
- **Domain knowledge**: Industry-specific terminology
- **Query examples**: Sample queries and expected results

## Benefits
- More accurate SQL generation
- Better understanding of business terms
- Consistent metric calculations
- Faster query generation

## 🗄️ Setup: Create Sample Database

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

conn = sqlite3.connect('company_metrics.db')

# Create users table
users = pd.DataFrame({
    'user_id': range(1, 201),
    'email': [f'user{i}@example.com' for i in range(1, 201)],
    'signup_date': [(datetime(2024, 1, 1) + timedelta(days=np.random.randint(0, 90))).strftime('%Y-%m-%d') for _ in range(200)],
    'plan': np.random.choice(['free', 'basic', 'pro', 'enterprise'], 200, p=[0.5, 0.3, 0.15, 0.05]),
    'country': np.random.choice(['US', 'UK', 'CA', 'AU', 'DE'], 200)
})

# Create subscriptions table
subscriptions = pd.DataFrame({
    'subscription_id': range(1, 101),
    'user_id': np.random.choice(range(1, 201), 100, replace=False),
    'start_date': [(datetime(2024, 1, 1) + timedelta(days=np.random.randint(0, 90))).strftime('%Y-%m-%d') for _ in range(100)],
    'mrr': np.random.choice([9.99, 29.99, 99.99, 299.99], 100),
    'status': np.random.choice(['active', 'cancelled', 'past_due'], 100, p=[0.85, 0.10, 0.05])
})

# Create events table
events = pd.DataFrame({
    'event_id': range(1, 1001),
    'user_id': np.random.randint(1, 201, 1000),
    'event_type': np.random.choice(['login', 'feature_use', 'upgrade', 'downgrade'], 1000),
    'timestamp': [(datetime(2024, 1, 1) + timedelta(hours=np.random.randint(0, 2160))).strftime('%Y-%m-%d %H:%M:%S') for _ in range(1000)]
})

users.to_sql('users', conn, if_exists='replace', index=False)
subscriptions.to_sql('subscriptions', conn, if_exists='replace', index=False)
events.to_sql('events', conn, if_exists='replace', index=False)

conn.close()
print("✅ Company metrics database created!")

In [ ]:
%%calliope add-database
{
    "datasource_id": "saas_metrics",
    "name": "SaaS Metrics Database",
    "dialect": "sqlite",
    "connection_details": {
        "database": "company_metrics.db"
    }
}

## 🔍 Before Training: Query Without Context

Let's try a business-specific query before RAG training:

In [ ]:
%%calliope ask-sql saas_metrics
What is our MRR and ARR?

**Note**: The model might struggle with business terms like MRR (Monthly Recurring Revenue) and ARR (Annual Recurring Revenue) without training.

## 📚 RAG Training: Teaching Business Logic

Now let's train Calliope with our schema, business definitions, and examples:

In [ ]:
%%calliope rag-train saas_metrics
{
    "ddl": [
        "CREATE TABLE users (user_id INTEGER PRIMARY KEY, email TEXT, signup_date DATE, plan TEXT, country TEXT)",
        "CREATE TABLE subscriptions (subscription_id INTEGER PRIMARY KEY, user_id INTEGER, start_date DATE, mrr REAL, status TEXT, FOREIGN KEY (user_id) REFERENCES users(user_id))",
        "CREATE TABLE events (event_id INTEGER PRIMARY KEY, user_id INTEGER, event_type TEXT, timestamp DATETIME, FOREIGN KEY (user_id) REFERENCES users(user_id))"
    ],
    "documentation": [
        "MRR (Monthly Recurring Revenue): Sum of all active subscription MRR values. Calculate as: SELECT SUM(mrr) FROM subscriptions WHERE status = 'active'",
        "ARR (Annual Recurring Revenue): MRR multiplied by 12. Calculate as: SELECT SUM(mrr) * 12 FROM subscriptions WHERE status = 'active'",
        "Churn Rate: Percentage of cancelled subscriptions. Calculate as: (cancelled_subscriptions / total_subscriptions) * 100",
        "Active Users: Users with at least one event in the last 30 days",
        "ARPU (Average Revenue Per User): Total MRR divided by number of active subscriptions",
        "Plan types: free (no charge), basic ($9.99), pro ($29.99), enterprise ($99.99/$299.99)",
        "Event types: login (user authentication), feature_use (feature interaction), upgrade (plan upgrade), downgrade (plan downgrade)"
    ],
    "sql": [
        "-- Get total MRR\nSELECT SUM(mrr) as total_mrr FROM subscriptions WHERE status = 'active';",
        "-- Get ARR\nSELECT SUM(mrr) * 12 as arr FROM subscriptions WHERE status = 'active';",
        "-- Calculate churn rate\nSELECT (COUNT(CASE WHEN status = 'cancelled' THEN 1 END) * 100.0 / COUNT(*)) as churn_rate FROM subscriptions;",
        "-- Get active users (30 days)\nSELECT COUNT(DISTINCT user_id) FROM events WHERE timestamp >= date('now', '-30 days');",
        "-- Calculate ARPU\nSELECT SUM(mrr) / COUNT(*) as arpu FROM subscriptions WHERE status = 'active';"
    ]
}

## ✅ After Training: Query With Context

Now let's try the same queries with our trained model:

In [ ]:
%%calliope ask-sql saas_metrics
What is our MRR and ARR?

In [ ]:
%%calliope ask-sql saas_metrics
Calculate our churn rate

In [ ]:
%%calliope ask-sql saas_metrics
What is our ARPU?

In [ ]:
%%calliope ask-sql saas_metrics
How many active users do we have?

## 🎯 Training with Complex Business Logic

In [ ]:
%%calliope rag-train saas_metrics
{
    "documentation": [
        "LTV (Lifetime Value): Average revenue per user divided by churn rate. Formula: ARPU / (Churn Rate / 100)",
        "CAC Payback Period: Time to recover customer acquisition cost. Calculate as months until subscription value exceeds acquisition cost.",
        "Net Revenue Retention (NRR): Revenue retention from existing customers including upgrades and downgrades. Formula: ((Starting MRR + Expansion - Contraction - Churn) / Starting MRR) * 100",
        "Quick Ratio: Growth efficiency metric. Formula: (New MRR + Expansion MRR) / (Churned MRR + Contraction MRR)",
        "Power Users: Users in the top 20% by feature_use events in the last 30 days"
    ],
    "sql": [
        "-- Identify power users\nWITH user_activity AS (SELECT user_id, COUNT(*) as activity_count FROM events WHERE event_type = 'feature_use' AND timestamp >= date('now', '-30 days') GROUP BY user_id)\nSELECT user_id FROM user_activity WHERE activity_count >= (SELECT PERCENTILE_CONT(0.8) WITHIN GROUP (ORDER BY activity_count) FROM user_activity);"
    ]
}

In [ ]:
%%calliope ask-sql saas_metrics --sql-model anthropic
Calculate our LTV (Lifetime Value)

In [ ]:
%%calliope ask-sql saas_metrics
Show me our power users and their activity levels

## 🔄 Update Schema Information

When your database schema changes, update Calliope:

In [ ]:
%%calliope update-schema saas_metrics

## 📋 View Current Training Data

You can verify what Calliope has learned by asking about specific metrics:

In [ ]:
%%calliope ask saas_metrics
How should I calculate MRR and ARR based on our database?

## 🗑️ Clear RAG Training Data

If you need to start fresh, clear all training data:

In [ ]:
%%calliope clear-rag saas_metrics

## 💡 Best Practices for RAG Training

### 1. Schema Documentation (DDL)
- Include table relationships (foreign keys)
- Document constraints and indexes
- Add column descriptions

### 2. Business Logic Documentation
- Define all custom metrics with formulas
- Explain business terms and abbreviations
- Document calculation methods
- Include edge cases and exclusions

### 3. SQL Examples
- Provide working queries for common questions
- Include complex joins and aggregations
- Show performance-optimized queries
- Add comments explaining the logic

### 4. Maintenance
- Update training when schema changes
- Review and improve based on query results
- Add examples for frequently asked questions
- Keep documentation current

## 🎯 Key Takeaways

- RAG training dramatically improves query accuracy
- Train with schema, documentation, and SQL examples
- Update regularly as your database evolves
- Include business-specific terminology and metrics
- More training data = better results

## 📚 Next Steps

- **Calliope-DirectSQL.ipynb** - Generate and edit SQL queries
- **Calliope-MultiDatabase.ipynb** - Manage multiple data sources